# Notebook 01 — Exploratory Data Analysis

**Project:** Fraud Detection & Strategy Analytics  
**Objective:** Understand the structure of the synthetic transaction dataset, identify key fraud patterns, and surface actionable insights to guide feature engineering and strategy design.

---

## Outline
1. Load & inspect data
2. Class distribution (fraud vs. legitimate)
3. Univariate distributions
4. Fraud rate by categorical segments
5. Fraud rate by time (hour, day)
6. Fraud rate by amount bucket
7. Correlation heatmap
8. Key insights summary

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from data.generate_data import generate_dataset

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded.')

## 1. Load Data

We generate the synthetic dataset on the fly using `generate_data.py`.
If you have already run `python data/generate_data.py`, you can swap the
cell below to `pd.read_csv('../data/transactions.csv')`.

In [ ]:
df = generate_dataset(n=100_000)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe().round(2)

## 2. Class Distribution

Fraud detection is a heavily imbalanced problem.
Our synthetic dataset mirrors the real world with ~2 % fraud prevalence.

In [ ]:
fraud_rate = df['is_fraud'].mean()
counts = df['is_fraud'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'],
            startangle=140)
axes[1].set_title('Class Balance')

plt.suptitle(f'Overall Fraud Rate: {fraud_rate:.2%}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Fraud transactions : {counts[1]:,}')
print(f'Legit transactions : {counts[0]:,}')
print(f'Fraud rate         : {fraud_rate:.2%}')

> **Insight:** The 2 % fraud rate means models trained without balancing will default to always predicting legitimate. We must use class weighting or resampling techniques.

## 3. Univariate Distributions — Fraud vs. Legitimate

In [ ]:
numeric_features = [
    'transaction_amount', 'time_of_day', 'user_age',
    'account_tenure_days', 'velocity_last_24h'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

fraud_df = df[df['is_fraud'] == 1]
legit_df = df[df['is_fraud'] == 0]

for i, col in enumerate(numeric_features):
    axes[i].hist(legit_df[col], bins=50, alpha=0.6, color='steelblue', label='Legitimate', density=True)
    axes[i].hist(fraud_df[col], bins=50, alpha=0.7, color='tomato',    label='Fraud',      density=True)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].legend(fontsize=9)

axes[-1].axis('off')  # hide unused subplot
plt.suptitle('Feature Distributions: Fraud vs Legitimate', fontsize=14)
plt.tight_layout()
plt.show()

> **Key observations:**
> - **transaction_amount**: Fraud transactions have a heavier right tail — larger amounts are disproportionately fraudulent.
> - **velocity_last_24h**: Fraudulent accounts show dramatically higher transaction velocity.
> - **account_tenure_days**: Newer accounts (low tenure) show higher fraud prevalence.
> - **time_of_day**: Fraud is more uniformly distributed across all hours; legitimate activity peaks during business hours.

## 4. Fraud Rate by Merchant Category

In [ ]:
cat_fraud = (
    df.groupby('merchant_category')['is_fraud']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'sum': 'fraud_count', 'count': 'total'})
    .sort_values('fraud_rate', ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_fraud['merchant_category'], cat_fraud['fraud_rate'] * 100,
               color=sns.color_palette('Reds_r', len(cat_fraud)))
ax.set_xlabel('Fraud Rate (%)')
ax.set_title('Fraud Rate by Merchant Category')
ax.axvline(fraud_rate * 100, color='navy', linestyle='--', label=f'Average ({fraud_rate:.1%})')
ax.legend()
for bar, rate in zip(bars, cat_fraud['fraud_rate']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print(cat_fraud.to_string(index=False))

> **Insight:** Travel and electronics categories carry the highest fraud risk — consistent with real-world patterns where fraudsters prefer high-value, easily resalable goods and services.

## 5. Fraud Rate by Time

In [ ]:
hourly = df.groupby('time_of_day')['is_fraud'].mean().reset_index()
daily  = df.groupby('day_of_week')['is_fraud'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly
axes[0].bar(hourly['time_of_day'], hourly['is_fraud'] * 100,
            color='steelblue', edgecolor='white')
axes[0].axhline(fraud_rate * 100, color='tomato', linestyle='--', label='Average')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Hour of Day')
axes[0].legend()

# Daily
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1].bar([day_labels[d] for d in daily['day_of_week']], daily['is_fraud'] * 100,
            color='mediumpurple', edgecolor='white')
axes[1].axhline(fraud_rate * 100, color='tomato', linestyle='--', label='Average')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Day of Week')
axes[1].legend()

plt.tight_layout()
plt.show()

> **Insight:** Late-night and early-morning hours (midnight–5 AM) show elevated fraud rates, as fraudsters operate when customer-service channels are less active and real cardholders are unlikely to be transacting.

## 6. Fraud Rate by Transaction Amount Bucket

In [ ]:
df['amount_bucket_label'] = pd.qcut(
    df['transaction_amount'], q=10,
    labels=[f'Q{i}' for i in range(1, 11)],
    duplicates='drop'
)

amt_fraud = (
    df.groupby('amount_bucket_label', observed=True)['is_fraud']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'count': 'n_transactions'})
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.bar(amt_fraud['amount_bucket_label'], amt_fraud['fraud_rate'] * 100,
        color='tomato', alpha=0.8, label='Fraud Rate (%)')
ax2.plot(amt_fraud['amount_bucket_label'], amt_fraud['n_transactions'],
         color='steelblue', marker='o', label='Transaction Volume')

ax1.set_xlabel('Amount Decile (Q1=Lowest, Q10=Highest)')
ax1.set_ylabel('Fraud Rate (%)', color='tomato')
ax2.set_ylabel('Transaction Count', color='steelblue')
ax1.set_title('Fraud Rate and Volume by Transaction Amount Decile')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

> **Insight:** Fraud rate rises monotonically with transaction amount. The top decile (largest amounts) carries 3–5× the average fraud rate — high-value transactions warrant extra scrutiny.

## 7. Correlation Heatmap

In [ ]:
numeric_df = df[[
    'transaction_amount', 'time_of_day', 'day_of_week', 'user_age',
    'account_tenure_days', 'previous_fraud_flag', 'location_mismatch',
    'velocity_last_24h', 'is_fraud'
]]

corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    square=True, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

> **Key correlations with `is_fraud`:**
> - `velocity_last_24h` — strong positive correlation (more transactions → higher fraud risk)
> - `location_mismatch` — positive correlation (unusual location is a fraud signal)
> - `previous_fraud_flag` — positive correlation (past behaviour predicts future)
> - `account_tenure_days` — negative correlation (newer accounts are riskier)
> - `transaction_amount` — moderate positive correlation

## 8. Velocity and Location Mismatch Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Velocity boxplot by fraud label
fraud_map = {0: 'Legitimate', 1: 'Fraud'}
df['label'] = df['is_fraud'].map(fraud_map)
sns.boxplot(data=df, x='label', y='velocity_last_24h',
            palette={'Legitimate': 'steelblue', 'Fraud': 'tomato'},
            ax=axes[0], showfliers=False)
axes[0].set_title('Transaction Velocity (last 24h) by Class')
axes[0].set_xlabel('')
axes[0].set_ylabel('Transaction count in last 24h')

# Location mismatch fraud rate
loc_fraud = df.groupby(['location_mismatch', 'label'])['transaction_id'].count().unstack()
loc_fraud_rate = df.groupby('location_mismatch')['is_fraud'].mean() * 100
axes[1].bar(['No Mismatch', 'Location Mismatch'], loc_fraud_rate.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[1].set_title('Fraud Rate: Location Mismatch')
axes[1].set_ylabel('Fraud Rate (%)')
for i, v in enumerate(loc_fraud_rate.values):
    axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## 9. Key Insights Summary

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **Travel & electronics** MCC codes have 3–5× average fraud rate | Apply stricter review thresholds for these categories |
| 2 | **Late-night hours** (midnight–5 AM) show elevated fraud | Consider step-up authentication for off-hours transactions |
| 3 | **High velocity** (>10 txns / 24h) is a strong fraud predictor | Hard-rule decline for velocity > 15 |
| 4 | **Location mismatch** + prior fraud = very high risk combo | Rule-based immediate decline for this combination |
| 5 | **Large amounts** (top decile) carry disproportionate fraud | Lower score cutoffs for high-value transactions |
| 6 | **New accounts** (<90 days tenure) are riskier | Apply conservative limits for new account period |
| 7 | The dataset is **~2% imbalanced** | Must use class weighting / SMOTE + optimise recall-focused metrics |

These insights directly inform the feature engineering in Notebook 02 and the strategy design in Notebook 04.